# Play A2C on Atari Assault
Load a trained A2C checkpoint and visualize one episode.

What this notebook does:
- Rebuild the same A2C architecture used during training.
- Automatically find and load the latest checkpoint from the A2C model folder.
- Run one Atari episode with the trained policy.
- Display game frames with OpenCV and print step-by-step reward statistics.

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.distributions as D
from ale_py.vector_env import AtariVectorEnv

In [ ]:
class A2CNetwork(nn.Module):
    def __init__(self, action_dim):
        super().__init__()
        self.shared_layer = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(7 * 7 * 64, 512),
            nn.ReLU()
        )
        self.actor_layer = nn.Linear(512, action_dim)
        self.critic_layer = nn.Linear(512, 1)

    def forward(self, x):
        shared_output = self.shared_layer(x)
        action_probs = torch.softmax(self.actor_layer(shared_output), dim=-1)
        state_value = self.critic_layer(shared_output)
        dist = D.Categorical(action_probs)
        return dist, state_value

In [ ]:
def find_latest_a2c_checkpoint(model_dir: Path):
    checkpoints = list(model_dir.glob('timesteps_*.pt'))
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint found in: {model_dir}')

    def extract_step(path: Path):
        name = path.stem  # e.g. a2c_1000000
        try:
            return int(name.split('_')[-1])
        except ValueError:
            return -1

    checkpoints.sort(key=extract_step)
    return checkpoints[-1]


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

model_dir = Path('..') / 'models' / 'a2c'
model_path = find_latest_a2c_checkpoint(model_dir)
print('Loading checkpoint:', model_path)

net = A2CNetwork(action_dim=7).to(device)
net.load_state_dict(torch.load(model_path, map_location=device))
net.eval()

## Checkpoint Loading Notes
This section locates the latest saved A2C checkpoint and loads it into the network.

The checkpoint selection function sorts files by training step number,
so the demo uses the most recent model automatically.

In [ ]:
envs = AtariVectorEnv(
    game='assault',
    num_envs=1,
    stack_num=4
)

In [ ]:
def run_env(envs, net, device, stochastic=False):
    observations, _ = envs.reset()
    total_rewards = 0.0
    step = 0

    while True:
        with torch.no_grad():
            obs_tensor = torch.tensor(observations, dtype=torch.float32, device=device) / 255.0
            dist, state_value = net(obs_tensor)
            if stochastic:
                actions = dist.sample()
            else:
                actions = torch.argmax(dist.probs, dim=1)
            actions = np.array(actions.cpu())

        observations, rewards, terminations, truncations, _ = envs.step(actions)

        step += 1
        total_rewards += float(rewards[0])

        frame = observations[0, 0, :, :].reshape(84, 84)
        frame_big = cv2.resize(
            frame,
            None,
            fx=4,
            fy=4,
            interpolation=cv2.INTER_NEAREST
        )
        cv2.imshow('obs', frame_big)

        print(
            f'step={step} | reward={float(rewards[0]):.2f} | total_reward={total_rewards:.2f} | V(s)={float(state_value[0].item()):.2f}',
            end='\r',
            flush=True
        )

        key = cv2.waitKey(100) & 0xFF
        if key == ord('q') or terminations[0] or truncations[0]:
            break

    print(f'\nEpisode done | steps={step} | total_reward={total_rewards:.2f}')
    cv2.destroyAllWindows()

In [ ]:
# stochastic=False: greedy action for stable demo
# stochastic=True : sample action from policy for exploratory demo
run_env(envs, net, device, stochastic=False)